# Chạy trọn quy trình trên video phòng thi (phase 2g)

Notebook này chạy `scripts/process_video.py` cho **mọi video `.mp4`** trong một Dataset của Kaggle. Mỗi video cho ra video đã đánh dấu, báo cáo và danh sách sự kiện.

**Làm lần lượt:**
1. Bên phải, mở **Settings**: bật **Internet** (Kaggle sẽ đòi xác minh số điện thoại một lần) và chọn **Accelerator → GPU T4 x2**.
2. Thêm video: **Add Input → Upload** thành một Dataset, đặt chế độ **Private**. Video thí sinh là dữ liệu cá nhân, không để công khai.
3. Nếu repo GitHub là private: **Add-ons → Secrets**, thêm secret tên `GITHUB_TOKEN` (token chỉ cần quyền đọc repo). Repo công khai thì bỏ qua.
4. Sửa các biến ở ô số 2: `DATASET_DIR` (thư mục Dataset của bạn, xem ở thanh bên phải mục Input), `BRANCH`, và `QUALITY` nếu cần.
5. Bấm **Run All** (hoặc Run → Run All). Chờ đến khi ô cuối in ra bảng tóm tắt.
6. Mở tab **Output** của notebook, tải file **`ket-qua.zip`** về máy. Trong đó mỗi video có một thư mục riêng (`annotated.mp4`, `report.html`, `events.csv`, ...).

Nếu một video bị lỗi, notebook ghi lỗi vào bảng và chạy tiếp video sau.

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

BRANCH = "claude/project-thread-g4frkm"
REPO = "github.com/vuthanhhnguyenn/httm"
DATASET_DIR = "/kaggle/input/<dataset>"  # sửa cho đúng tên Dataset của bạn
QUALITY = "max"  # đổi thành "balanced" nếu bảng tóm tắt có tỉ lệ lớn hơn 2 (chạy quá chậm)
WORK = Path("/kaggle/working")
REPO_DIR = WORK / "httm"
OUT_ROOT = WORK / "ket-qua"

token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # không có secret: clone bằng URL công khai

def sh(cmd, cwd=None, env=None):
    print("$", cmd)
    try:
        subprocess.run(cmd, shell=True, check=True, cwd=cwd, env=env)
    except subprocess.CalledProcessError as exc:
        # Re-raise without the chained traceback so nothing but the command (never the token) is shown.
        raise RuntimeError(f"Lệnh lỗi (mã {exc.returncode}): {cmd}") from None

VIDEOS = sorted(Path(DATASET_DIR).rglob("*.mp4")) if Path(DATASET_DIR).is_dir() else []
assert VIDEOS, f"Không thấy file .mp4 nào trong {DATASET_DIR} (sửa biến DATASET_DIR)"
print(f"Tìm thấy {len(VIDEOS)} video:", *[v.name for v in VIDEOS], sep="\n  ")
print("Python", __import__("sys").version.split()[0])
assert __import__("sys").version_info >= (3, 11), "Cần Python 3.11 trở lên"
sh("nvidia-smi -L")

In [ ]:
# Clone nhánh và cài những gói còn thiếu (Kaggle đã có torch, opencv, numpy). Repo không đóng gói pip nên
# các script tự thêm packages/*/src vào sys.path; không cần pip install -e.
# Token goes only through environment variables (git's GIT_CONFIG_* ), never the URL, argv or .git/config.
env = dict(os.environ)
if token:
    import base64
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
               GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}")
if not REPO_DIR.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} https://{REPO}.git {REPO_DIR}", env=env)
else:
    # Chạy lại trong cùng phiên: lấy commit mới nhất của nhánh thay vì dùng bản cũ đã clone.
    sh(f"git fetch --depth 1 origin {BRANCH}", cwd=REPO_DIR, env=env)
    sh("git reset --hard FETCH_HEAD", cwd=REPO_DIR)
sh('pip install -q "ultralytics==8.4.160" pyyaml "pydantic>=2.9,<3.0"')

In [ ]:
# Tải model (kiểm SHA-256 theo models/manifest.yaml)
sh("python scripts/download_model.py", cwd=REPO_DIR)

In [ ]:
# Chạy từng video. Lỗi ở một video chỉ ghi vào bảng, các video còn lại vẫn chạy.
import torch

GPUS = torch.cuda.device_count()
DEVICES = "--device 0 --device2 1" if GPUS >= 2 else "--device 0"
print(f"Số GPU: {GPUS} -> dùng {DEVICES}; chất lượng: {QUALITY}")

OUT_ROOT.mkdir(parents=True, exist_ok=True)
errors = {}
for index, video in enumerate(VIDEOS, 1):
    out = OUT_ROOT / video.stem
    print(f"\n===== [{index}/{len(VIDEOS)}] {video.name} =====")
    try:
        sh(f'python scripts/process_video.py "{video}" --out "{out}" --quality {QUALITY} {DEVICES}', cwd=REPO_DIR)
    except Exception as exc:  # noqa: BLE001 - keep going with the next video
        errors[video.stem] = str(exc)
        print(f"LỖI với {video.name}: {exc}")

In [ ]:
# Bảng tóm tắt: đọc timing.json và events.jsonl của từng video (chỉ đếm bản ghi action == "opened")
rows = []
for video in VIDEOS:
    stem = video.stem
    row = {"video": stem, "thời lượng (s)": None, "thời gian chạy (s)": None, "tỉ lệ": None,
           "đỏ": None, "vàng": None, "vượt mốc": None, "lỗi": errors.get(stem, "")}
    out = OUT_ROOT / stem
    try:
        timing = json.loads((out / "timing.json").read_text(encoding="utf-8"))
        row.update({"thời lượng (s)": timing["video_s"], "thời gian chạy (s)": timing["total_s"],
                    "tỉ lệ": timing["ratio"], "vượt mốc": "CÓ" if timing.get("over_budget") else "không"})
        records = [json.loads(line) for line in (out / "events.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
        opened = [r for r in records if r.get("action") == "opened"]
        row["đỏ"] = sum(1 for r in opened if r.get("level") == "red")
        row["vàng"] = len(opened) - row["đỏ"]
    except Exception as exc:  # noqa: BLE001 - a failed video has no outputs
        row["lỗi"] = row["lỗi"] or f"không đọc được kết quả: {exc}"
    rows.append(row)

try:
    import pandas as pd
    pd.set_option("display.width", 200)
    pd.set_option("display.max_colwidth", 80)
    print(pd.DataFrame(rows).to_string(index=False))
except ImportError:
    cols = list(rows[0])
    print(" | ".join(cols))
    for r in rows:
        print(" | ".join("" if r[c] is None else str(r[c]) for c in cols))
if any(r["vượt mốc"] == "CÓ" for r in rows):
    print("\nCó video vượt mốc 2× thời lượng: đổi QUALITY = \"balanced\" ở ô 2 rồi chạy lại.")

In [ ]:
# Nén kết quả để tải về
archive = shutil.make_archive(str(WORK / "ket-qua"), "zip", root_dir=WORK, base_dir="ket-qua")
size_mb = Path(archive).stat().st_size / 1e6
print(f"Đã nén: {archive} ({size_mb:.0f} MB)")
print("Tải về: mở tab Output của notebook (bên phải), chọn ket-qua.zip rồi bấm Download.")